In [1]:
# Intraday volatility/activity strategy (daily filter + 3-minute execution)
#
# Daily filters:
# - RVol > 2
# - ATRs_Traded > 2
# - Close > Open
#
# 3-minute entry:
# - Close < VWAP
# - EMA(5) crosses above EMA(9)
#
# 3-minute exit:
# - EMA(5) crosses below EMA(9)
#
# Stop:
# - Low of day

In [2]:
# Setup: imports + optional dependency installs
import sys
import subprocess
import importlib
from collections.abc import Iterable


def ensure_package(pip_name: str, import_name: str | None = None) -> None:
    import_name = import_name or pip_name
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        importlib.import_module(import_name)


ensure_package("pandas")
ensure_package("numpy")
ensure_package("backtrader")

import backtrader as bt
import numpy as np
import pandas as pd

from strategies.data import add_market_data_to_syspath, load_daily_variables

# Make sibling market_data package importable.
add_market_data_to_syspath()

# Required by your instruction: intraday data must come from intraday_import.
try:
    from market_data.price_data_import import intraday_import
except ImportError:
    from price_data_import import intraday_import

In [3]:
# Parameters and data loading
DAILY_VARIABLES_PATH = None  # e.g. r"C:\\path\\to\\daily_variables.pkl"
DAILY_VARIABLES_FILENAME = "daily_variables.pkl"

TARGET_INTRADAY_MINUTES = 3
INTRADAY_RESAMPLE = "3min"  # resample output from intraday_import
INTRADAY_TIMESPAN = "minute"
INTRADAY_MULTIPLIER = 1
INTRADAY_LIMIT = 50_000

# Backtest window for intraday pull
LOOKBACK_DAYS = 20
END_DATE = pd.Timestamp.today().normalize()
START_DATE = END_DATE - pd.Timedelta(days=LOOKBACK_DAYS)

RVol_THRESHOLD = 2.0
ATRS_TRADED_THRESHOLD = 2.0
FLAG_DAYS_AFTER_SIGNAL = 5  # signal day + N following trading days are entry-eligible
MAX_SYMBOLS = None  # set an int while iterating (for speed), e.g. 50
TRADE_SIZE = 1
COMMISSION = 0.001
INITIAL_CASH = 100_000.0

if DAILY_VARIABLES_PATH:
    from strategies.data import load_pickled_variables

    symbols = load_pickled_variables(DAILY_VARIABLES_PATH)
else:
    symbols = load_daily_variables(filename=DAILY_VARIABLES_FILENAME)

if not isinstance(symbols, dict):
    raise TypeError(f"Expected `symbols` to be a dict, got: {type(symbols)!r}")

universe = sorted(symbols.keys())
if MAX_SYMBOLS is not None:
    universe = universe[: int(MAX_SYMBOLS)]

print(f"Universe size: {len(universe):,}")
print(f"Intraday window: {START_DATE.date()} -> {END_DATE.date()} | resample={INTRADAY_RESAMPLE}")

Loading Variables: 100%|██████████| 26/26 [01:58<00:00,  4.57s/it]


Universe size: 2,963
Intraday window: 2026-01-27 -> 2026-02-16 | resample=3min


In [4]:
# Helpers: dataframe discovery, normalization, and frame selection
RVOL_CANDIDATES = ["rvol", "relative_volume", "rel_volume"]
ATRS_TRADED_CANDIDATES = ["atrs_traded", "atrs_traded_14", "atrs_trade"]
VWAP_CANDIDATES = ["vwap", "session_vwap", "day_vwap", "intraday_vwap"]


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    return df.rename(columns={c: str(c).strip().lower().replace(" ", "_") for c in df.columns})


def ensure_datetime_index(df: pd.DataFrame, *, symbol: str, frame_name: str) -> pd.DataFrame:
    out = df.copy()
    if not isinstance(out.index, pd.DatetimeIndex):
        if "date" in out.columns:
            out["date"] = pd.to_datetime(out["date"], errors="coerce")
            out = out.set_index("date")
        elif "datetime" in out.columns:
            out["datetime"] = pd.to_datetime(out["datetime"], errors="coerce")
            out = out.set_index("datetime")
        else:
            raise TypeError(f"{symbol}: {frame_name} must have a DatetimeIndex (or date/datetime column).")

    out.index = pd.to_datetime(out.index, errors="coerce")
    out = out[~out.index.isna()].copy()

    if getattr(out.index, "tz", None) is not None:
        out.index = out.index.tz_convert(None)

    return out.sort_index()


def to_numeric(df: pd.DataFrame, columns: Iterable[str]) -> pd.DataFrame:
    out = df.copy()
    for col in columns:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def pick_first_present(df: pd.DataFrame, candidates: list[str], *, label: str, symbol: str) -> str:
    col = next((c for c in candidates if c in df.columns), None)
    if col is None:
        raise ValueError(f"{symbol}: missing {label}. Candidates: {candidates}")
    return col


def infer_minutes_per_bar(index: pd.DatetimeIndex) -> float | None:
    if len(index) < 3:
        return None

    deltas = pd.Series(index).diff().dropna().dt.total_seconds() / 60.0
    deltas = deltas[deltas > 0]
    if deltas.empty:
        return None
    return float(deltas.median())


def collect_dataframe_candidates(obj, *, root_name: str, max_depth: int = 2):
    candidates: list[tuple[str, pd.DataFrame]] = []
    visited: set[int] = set()

    def _walk(value, name: str, depth: int) -> None:
        if depth > max_depth:
            return

        obj_id = id(value)
        if obj_id in visited:
            return
        visited.add(obj_id)

        if isinstance(value, pd.DataFrame):
            if not value.empty:
                candidates.append((name, value))
            return

        if isinstance(value, dict):
            for k, v in value.items():
                _walk(v, f"{name}.{k}", depth + 1)
            return

        if hasattr(value, "df"):
            try:
                df_value = getattr(value, "df")
            except Exception:
                df_value = None
            if isinstance(df_value, pd.DataFrame) and not df_value.empty:
                candidates.append((f"{name}.df", df_value))

        if depth == max_depth:
            return

        for attr in dir(value):
            if attr.startswith("_"):
                continue
            try:
                attr_value = getattr(value, attr)
            except Exception:
                continue
            if callable(attr_value):
                continue
            if isinstance(attr_value, (pd.DataFrame, dict)) or hasattr(attr_value, "df"):
                _walk(attr_value, f"{name}.{attr}", depth + 1)

    _walk(obj, root_name, 0)

    unique: list[tuple[str, pd.DataFrame]] = []
    seen: set[int] = set()
    for name, frame in candidates:
        frame_id = id(frame)
        if frame_id in seen:
            continue
        seen.add(frame_id)
        unique.append((name, frame))

    return unique


def select_daily_and_intraday_frames(candidates, *, target_minutes: int = 3):
    scored = []
    for name, frame in candidates:
        try:
            idx = frame.index if isinstance(frame.index, pd.DatetimeIndex) else pd.to_datetime(frame.index, errors="coerce")
        except Exception:
            idx = None

        if idx is None:
            minutes = None
            bars = 0
        else:
            valid_idx = idx[~pd.isna(idx)]
            bars = int(len(valid_idx))
            minutes = infer_minutes_per_bar(pd.DatetimeIndex(valid_idx)) if bars > 0 else None

        lname = name.lower()
        intraday_hint = int(any(tok in lname for tok in ["3m", "3_min", "3min", "intraday", "minute", "min"]))
        daily_hint = int(any(tok in lname for tok in ["daily", "eod", "day"]))

        intraday_freq = int(minutes is not None and minutes <= 30)
        daily_freq = int(minutes is not None and minutes >= 60)

        closeness = -abs((minutes if minutes is not None else 10_000.0) - float(target_minutes))

        scored.append(
            {
                "name": name,
                "frame": frame,
                "bars": bars,
                "minutes": minutes,
                "intraday_rank": (intraday_hint, intraday_freq, closeness, bars),
                "daily_rank": (daily_hint, daily_freq, bars),
            }
        )

    if not scored:
        raise ValueError("No dataframe candidates found")

    intraday_pick = sorted(scored, key=lambda x: x["intraday_rank"], reverse=True)[0]
    daily_pick = sorted(scored, key=lambda x: x["daily_rank"], reverse=True)[0]

    return daily_pick, intraday_pick


def prepare_symbol_context(symbol: str, symbol_data, *, target_intraday_minutes: int = 3):
    candidates = collect_dataframe_candidates(symbol_data, root_name=symbol)
    if not candidates:
        raise ValueError(f"{symbol}: no dataframe candidates discovered")

    daily_pick, intraday_pick = select_daily_and_intraday_frames(candidates, target_minutes=target_intraday_minutes)

    daily_df = ensure_datetime_index(daily_pick["frame"], symbol=symbol, frame_name=daily_pick["name"])
    intraday_df = ensure_datetime_index(intraday_pick["frame"], symbol=symbol, frame_name=intraday_pick["name"])

    daily_df = normalize_columns(daily_df)
    intraday_df = normalize_columns(intraday_df)

    if "close" not in daily_df.columns and "adj_close" in daily_df.columns:
        daily_df["close"] = daily_df["adj_close"]
    if "close" not in intraday_df.columns and "adj_close" in intraday_df.columns:
        intraday_df["close"] = intraday_df["adj_close"]

    # Ensure required fields for the strategy logic.
    for required_col in ["open", "close"]:
        if required_col not in daily_df.columns:
            raise ValueError(f"{symbol}: daily frame missing required column `{required_col}`")

    for required_col in ["open", "high", "low", "close"]:
        if required_col not in intraday_df.columns:
            raise ValueError(f"{symbol}: intraday frame missing required column `{required_col}`")

    daily_df = to_numeric(daily_df, [c for c in ["open", "close"] if c in daily_df.columns])
    intraday_df = to_numeric(intraday_df, [c for c in ["open", "high", "low", "close"] if c in intraday_df.columns])

    rvol_col = pick_first_present(daily_df, RVOL_CANDIDATES, label="RVol column", symbol=symbol)
    atrs_traded_col = pick_first_present(daily_df, ATRS_TRADED_CANDIDATES, label="ATRs_Traded column", symbol=symbol)
    vwap_col = pick_first_present(intraday_df, VWAP_CANDIDATES, label="VWAP column", symbol=symbol)

    daily_df = to_numeric(daily_df, [rvol_col, atrs_traded_col]).dropna(subset=["open", "close", rvol_col, atrs_traded_col])
    intraday_df = to_numeric(intraday_df, [vwap_col]).dropna(subset=["open", "high", "low", "close", vwap_col])

    return {
        "symbol": symbol,
        "daily_name": daily_pick["name"],
        "intraday_name": intraday_pick["name"],
        "daily_df": daily_df,
        "intraday_df": intraday_df,
        "rvol_col": rvol_col,
        "atrs_traded_col": atrs_traded_col,
        "vwap_col": vwap_col,
    }

In [5]:
# Daily filtering + Backtrader strategy implementation
RVOL_CANDIDATES = ["rvol", "relative_volume", "rel_volume"]
ATRS_TRADED_CANDIDATES = ["atrs_traded", "atrs_traded_14", "atrs_trade"]


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    return df.rename(columns={c: str(c).strip().lower().replace(" ", "_") for c in df.columns})


def prepare_daily_frame(symbol_data, symbol: str) -> tuple[pd.DataFrame, str, str]:
    if not hasattr(symbol_data, "df"):
        raise ValueError(f"{symbol}: SymbolData has no .df")

    df = symbol_data.df.copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        if "date" in df.columns:
            df["date"] = pd.to_datetime(df["date"], errors="coerce")
            df = df.set_index("date")
        else:
            raise TypeError(f"{symbol}: daily frame needs DatetimeIndex (or `date` column)")

    if getattr(df.index, "tz", None) is not None:
        df.index = df.index.tz_convert(None)

    df = normalize_columns(df).sort_index()

    if "close" not in df.columns and "adj_close" in df.columns:
        df["close"] = df["adj_close"]

    required = ["open", "close"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{symbol}: missing required daily columns: {missing}")

    rvol_col = next((c for c in RVOL_CANDIDATES if c in df.columns), None)
    atrs_col = next((c for c in ATRS_TRADED_CANDIDATES if c in df.columns), None)
    if rvol_col is None or atrs_col is None:
        raise ValueError(
            f"{symbol}: missing RVol/ATRs_Traded columns. "
            f"RVol candidates={RVOL_CANDIDATES}, ATR candidates={ATRS_TRADED_CANDIDATES}"
        )

    numeric_cols = ["open", "close", rvol_col, atrs_col]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=numeric_cols)
    return df, rvol_col, atrs_col


def qualifying_daily_mask(
    daily_df: pd.DataFrame,
    *,
    rvol_col: str,
    atrs_col: str,
    rvol_threshold: float,
    atrs_threshold: float,
) -> pd.Series:
    return (
        (daily_df[rvol_col] > float(rvol_threshold))
        & (daily_df[atrs_col] > float(atrs_threshold))
        & (daily_df["close"] > daily_df["open"])
    )


def build_flagged_entry_days(
    daily_index: pd.DatetimeIndex,
    signal_index: pd.DatetimeIndex,
    *,
    days_after_signal: int,
) -> tuple[tuple[pd.Timestamp, ...], dict[pd.Timestamp, pd.Timestamp]]:
    """Create entry-eligible day flags from each daily signal day.

    Flag window = signal day + next `days_after_signal` trading days.
    Returns:
      - sorted tuple of flagged entry days
      - mapping entry_day -> originating signal_day
    """
    if len(signal_index) == 0:
        return tuple(), {}

    # Work on unique normalized trading days present in daily data.
    trading_days = pd.Index(pd.to_datetime(daily_index).normalize().unique()).sort_values()
    day_to_pos = {pd.Timestamp(day): i for i, day in enumerate(trading_days)}

    flagged: set[pd.Timestamp] = set()
    entry_to_signal: dict[pd.Timestamp, pd.Timestamp] = {}

    for signal_ts in signal_index:
        signal_day = pd.Timestamp(signal_ts).normalize()
        pos = day_to_pos.get(signal_day)
        if pos is None:
            continue

        end_pos = min(pos + int(days_after_signal), len(trading_days) - 1)
        for i in range(pos, end_pos + 1):
            entry_day = pd.Timestamp(trading_days[i])
            flagged.add(entry_day)
            entry_to_signal.setdefault(entry_day, signal_day)

    flagged_sorted = tuple(sorted(flagged))
    return flagged_sorted, entry_to_signal


class IntradayPandasData(bt.feeds.PandasData):
    lines = ("vwap",)
    params = (
        ("datetime", None),
        ("open", "Open"),
        ("high", "High"),
        ("low", "Low"),
        ("close", "Close"),
        ("volume", "Volume"),
        ("openinterest", -1),
        ("vwap", "VWAP"),
    )


class TradeCaptureAnalyzer(bt.Analyzer):
    def start(self) -> None:
        self.trades: list[dict] = []

    @staticmethod
    def _safe_history_value(trade, index: int, attr: str):
        try:
            return float(getattr(trade.history[index].event, attr))
        except Exception:
            return None

    def notify_trade(self, trade) -> None:
        if not trade.isclosed:
            return

        strategy = self.strategy
        symbol = trade.data._name or "UNKNOWN"
        entry_dt = bt.num2date(trade.dtopen)
        exit_dt = bt.num2date(trade.dtclose)
        entry_day = entry_dt.date()

        metrics = strategy.daily_metrics_by_date.get(entry_day, {})
        stop_low = strategy.day_low_by_date.get(entry_day)

        self.trades.append(
            {
                "symbol": symbol,
                "trade_day": pd.Timestamp(entry_day),
                "entry_time": pd.Timestamp(entry_dt),
                "exit_time": pd.Timestamp(exit_dt),
                "entry_price": self._safe_history_value(trade, 0, "price"),
                "exit_price": self._safe_history_value(trade, -1, "price"),
                "size": self._safe_history_value(trade, 0, "size"),
                "pnl": float(trade.pnlcomm),
                "return_pct": (
                    (self._safe_history_value(trade, -1, "price") / self._safe_history_value(trade, 0, "price") - 1.0) * 100.0
                    if self._safe_history_value(trade, 0, "price")
                    else np.nan
                ),
                "exit_reason": strategy.last_exit_reason,
                "signal_day": metrics.get("signal_day"),
                "daily_rvol": metrics.get("rvol", np.nan),
                "daily_atrs_traded": metrics.get("atrs_traded", np.nan),
                "stop_low_of_day": stop_low,
            }
        )

    def get_analysis(self):
        return self.trades


class IntradayVolatilityActivityStrategy(bt.Strategy):
    params = dict(
        qualifying_days=(),
        day_low_by_date=None,
        daily_metrics_by_date=None,
        trade_size=1,
        close_at_session_end=True,
    )

    def __init__(self) -> None:
        self.ema5 = bt.ind.EMA(self.data.close, period=5)
        self.ema9 = bt.ind.EMA(self.data.close, period=9)
        self.cross = bt.ind.CrossOver(self.ema5, self.ema9)

        self.qualifying_days = {pd.Timestamp(d).date() for d in self.p.qualifying_days}
        self.day_low_by_date = {
            pd.Timestamp(k).date(): float(v) for k, v in (self.p.day_low_by_date or {}).items()
        }
        self.daily_metrics_by_date = {
            pd.Timestamp(k).date(): v for k, v in (self.p.daily_metrics_by_date or {}).items()
        }

        self.pending_order = None
        self.entry_day = None
        self.last_exit_reason = None

    def _is_last_bar_of_session(self) -> bool:
        current_day = self.data.datetime.date(0)
        try:
            next_day = self.data.datetime.date(1)
        except Exception:
            return True
        return next_day != current_day

    def notify_order(self, order) -> None:
        if order.status in [order.Submitted, order.Accepted]:
            return
        if order.status in [order.Canceled, order.Margin, order.Rejected, order.Completed]:
            self.pending_order = None

    def next(self) -> None:
        if self.pending_order is not None:
            return

        dt = self.data.datetime.datetime(0)
        trade_day = dt.date()

        # Entry: daily filter day + close < VWAP + EMA(5) cross above EMA(9)
        if not self.position:
            if trade_day not in self.qualifying_days:
                return
            if self.cross[0] > 0 and float(self.data.close[0]) < float(self.data.vwap[0]):
                self.entry_day = trade_day
                self.last_exit_reason = None
                self.pending_order = self.buy(size=int(self.p.trade_size))
            return

        active_day = self.entry_day or trade_day
        stop_price = self.day_low_by_date.get(active_day)

        # Stop: low of day
        if stop_price is not None and float(self.data.low[0]) <= float(stop_price):
            self.last_exit_reason = "stop_low_of_day"
            self.pending_order = self.close()
            return

        # Exit: EMA(5) crosses below EMA(9)
        if self.cross[0] < 0:
            self.last_exit_reason = "ema5_cross_below_ema9"
            self.pending_order = self.close()
            return

        # Keep this strictly intraday.
        if self.p.close_at_session_end and self._is_last_bar_of_session():
            self.last_exit_reason = "end_of_day"
            self.pending_order = self.close()

In [6]:
# Daily scan -> intraday_import pull -> Backtrader runs
window_start = pd.Timestamp(START_DATE).normalize()
window_end = pd.Timestamp(END_DATE).normalize()

scan_rows: list[dict] = []
skipped_symbols: list[dict] = []
daily_contexts: dict[str, dict] = {}

for sym in universe:
    try:
        daily_df, rvol_col, atrs_col = prepare_daily_frame(symbols[sym], sym)
        window_df = daily_df.loc[(daily_df.index.normalize() >= window_start) & (daily_df.index.normalize() <= window_end)].copy()

        if window_df.empty:
            scan_rows.append(
                {
                    "symbol": sym,
                    "signal_days": 0,
                    "flagged_days": 0,
                    "latest_qualifies": False,
                    "latest_flag_active": False,
                    "daily_rows_in_window": 0,
                }
            )
            continue

        mask = qualifying_daily_mask(
            window_df,
            rvol_col=rvol_col,
            atrs_col=atrs_col,
            rvol_threshold=RVol_THRESHOLD,
            atrs_threshold=ATRS_TRADED_THRESHOLD,
        )

        signal_idx = window_df.index[mask]
        signal_days = tuple(pd.Timestamp(ts).normalize() for ts in signal_idx)

        flagged_days, entry_to_signal_day = build_flagged_entry_days(
            window_df.index,
            signal_idx,
            days_after_signal=FLAG_DAYS_AFTER_SIGNAL,
        )

        # Carry originating daily signal metrics into each flagged entry day.
        daily_metrics_by_date = {}
        for entry_day, signal_day in entry_to_signal_day.items():
            signal_ts = pd.Timestamp(signal_day)
            if signal_ts not in window_df.index:
                continue

            signal_row = window_df.loc[signal_ts]
            if isinstance(signal_row, pd.DataFrame):
                signal_row = signal_row.iloc[-1]

            daily_metrics_by_date[pd.Timestamp(entry_day).date()] = {
                "rvol": float(signal_row[rvol_col]),
                "atrs_traded": float(signal_row[atrs_col]),
                "signal_day": signal_ts.date().isoformat(),
            }

        qualifying_days = tuple(pd.Timestamp(ts).date() for ts in flagged_days)

        latest_idx = window_df.index.max()
        latest_qualifies = bool(mask.loc[latest_idx]) if latest_idx in mask.index else False
        latest_flag_active = pd.Timestamp(latest_idx).normalize() in set(flagged_days)

        daily_contexts[sym] = {
            "qualifying_days": qualifying_days,
            "signal_days": tuple(pd.Timestamp(ts).date() for ts in signal_days),
            "daily_metrics_by_date": daily_metrics_by_date,
        }

        scan_rows.append(
            {
                "symbol": sym,
                "signal_days": int(mask.sum()),
                "flagged_days": int(len(flagged_days)),
                "latest_qualifies": latest_qualifies,
                "latest_flag_active": latest_flag_active,
                "latest_day": latest_idx.date().isoformat(),
                "daily_rows_in_window": int(len(window_df)),
            }
        )
    except Exception as exc:
        skipped_symbols.append({"symbol": sym, "reason": str(exc)})

scan_df = pd.DataFrame(scan_rows).sort_values(["latest_flag_active", "flagged_days", "symbol"], ascending=[False, False, True])
display(scan_df.head(30))

if skipped_symbols:
    print(f"Skipped symbols during daily prep: {len(skipped_symbols):,}")
    display(pd.DataFrame(skipped_symbols).head(20))

candidate_symbols = [s for s, ctx in daily_contexts.items() if len(ctx["qualifying_days"]) > 0]
print(f"Symbols with at least one flagged entry day in window: {len(candidate_symbols):,}")

if not candidate_symbols:
    raise ValueError("No qualifying symbols found in the selected lookback window.")

# Pull intraday prices using required function from price_data_import.
intraday_data = intraday_import(
    wl=candidate_symbols,
    from_date=window_start.date().isoformat(),
    to_date=window_end.date().isoformat(),
    resample=INTRADAY_RESAMPLE,
    timespan=INTRADAY_TIMESPAN,
    multiplier=INTRADAY_MULTIPLIER,
    limit=INTRADAY_LIMIT,
    market_open_only=True,
)

print(f"Intraday frames returned: {len(intraday_data):,}")


def prepare_intraday_frame(df_raw: pd.DataFrame, symbol: str) -> pd.DataFrame:
    if df_raw is None or df_raw.empty:
        raise ValueError(f"{symbol}: empty intraday frame")

    df = df_raw.copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        if "Timestamp" in df.columns:
            df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
            df = df.set_index("Timestamp")
        elif "timestamp" in df.columns:
            df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
            df = df.set_index("timestamp")
        else:
            raise TypeError(f"{symbol}: intraday frame needs DatetimeIndex or Timestamp column")

    if getattr(df.index, "tz", None) is not None:
        df.index = df.index.tz_convert(None)

    rename_map = {
        "open": "Open",
        "high": "High",
        "low": "Low",
        "close": "Close",
        "volume": "Volume",
        "vwap": "VWAP",
    }
    for src, dst in rename_map.items():
        if src in df.columns and dst not in df.columns:
            df[dst] = df[src]

    required = ["Open", "High", "Low", "Close", "Volume", "VWAP"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{symbol}: missing intraday columns: {missing}")

    for col in required:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.sort_index().dropna(subset=required)
    return df[required]


run_rows: list[dict] = []
trade_frames: list[pd.DataFrame] = []

for sym in candidate_symbols:
    if sym not in intraday_data:
        run_rows.append({"symbol": sym, "status": "missing_intraday_data"})
        continue

    try:
        intraday_df = prepare_intraday_frame(intraday_data[sym], sym)

        ctx = daily_contexts[sym]
        qualifying_days = set(ctx["qualifying_days"])
        if not qualifying_days:
            run_rows.append({"symbol": sym, "status": "no_qualifying_days"})
            continue

        # Keep intraday bars only on days passing the daily filter.
        day_mask = pd.Index(intraday_df.index.date).isin(qualifying_days)
        intraday_df = intraday_df.loc[day_mask].copy()

        if intraday_df.empty:
            run_rows.append({"symbol": sym, "status": "no_intraday_on_qualifying_days"})
            continue

        day_low_by_date = intraday_df.groupby(intraday_df.index.date)["Low"].min().to_dict()

        cerebro = bt.Cerebro(stdstats=False)
        cerebro.broker.setcash(INITIAL_CASH)
        cerebro.broker.setcommission(commission=COMMISSION)

        feed = IntradayPandasData(dataname=intraday_df)
        cerebro.adddata(feed, name=sym)
        cerebro.addstrategy(
            IntradayVolatilityActivityStrategy,
            qualifying_days=tuple(qualifying_days),
            day_low_by_date=day_low_by_date,
            daily_metrics_by_date=ctx["daily_metrics_by_date"],
            trade_size=TRADE_SIZE,
        )
        cerebro.addanalyzer(TradeCaptureAnalyzer, _name="trade_capture")

        start_value = float(cerebro.broker.getvalue())
        result = cerebro.run()[0]
        end_value = float(cerebro.broker.getvalue())

        trade_rows = result.analyzers.trade_capture.get_analysis()
        if trade_rows:
            trade_frames.append(pd.DataFrame(trade_rows))

        run_rows.append(
            {
                "symbol": sym,
                "status": "ok",
                "bars": int(len(intraday_df)),
                "trades": int(len(trade_rows)),
                "start_value": start_value,
                "end_value": end_value,
                "net_pnl": end_value - start_value,
            }
        )
    except Exception as exc:
        run_rows.append({"symbol": sym, "status": "error", "reason": str(exc)})

run_df = pd.DataFrame(run_rows).sort_values(["status", "symbol"]).reset_index(drop=True)
display(run_df.head(40))

if trade_frames:
    trades_df = pd.concat(trade_frames, ignore_index=True).sort_values(["entry_time", "symbol"]).reset_index(drop=True)

    summary = (
        trades_df.groupby("symbol", as_index=False)
        .agg(
            trades=("pnl", "count"),
            wins=("pnl", lambda x: int((x > 0).sum())),
            total_pnl=("pnl", "sum"),
            avg_pnl=("pnl", "mean"),
            avg_return_pct=("return_pct", "mean"),
        )
        .sort_values("total_pnl", ascending=False)
        .reset_index(drop=True)
    )
    summary["win_rate"] = (summary["wins"] / summary["trades"]) * 100.0

    print(f"Total trades: {len(trades_df):,}")
    display(summary.head(20))
    display(trades_df.head(30))
else:
    trades_df = pd.DataFrame()
    summary = pd.DataFrame()
    print("No trades were generated in Backtrader with the current settings.")

,symbol,qualifying_days,latest_qualifies,latest_day,daily_rows_in_window
1084,FSLY,2,True,2026-02-13,14
1693,MGA,2,True,2026-02-13,14
1709,MITK,2,True,2026-02-13,14
377,BIO,1,True,2026-02-13,14
586,CIM,1,True,2026-02-13,14
597,CLMT,1,True,2026-02-13,14
645,COHU,1,True,2026-02-13,14
646,COIN,1,True,2026-02-13,14
672,CPS,1,True,2026-02-13,14
695,CRSR,1,True,2026-02-13,14


Symbols with at least one qualifying day in window: 420


Importing Price Data: 100%|██████████| 420/420 [01:47<00:00,  3.90it/s]


Intraday frames returned: 420


,symbol,status,bars,trades,start_value,end_value,net_pnl
0,AAOI,ok,260,2,100000.0,100002.910280,2.910280
1,ABBV,ok,130,1,100000.0,99997.696675,-2.303325
2,ABUS,ok,119,0,100000.0,99999.999614,-0.000386
3,ACHC,ok,130,0,100000.0,100000.000000,0.000000
4,ACT,ok,128,1,100000.0,100000.792280,0.792280
5,ADT,ok,130,0,100000.0,100000.000000,0.000000
6,AEHR,ok,130,0,100000.0,100000.000000,0.000000
7,AES,ok,130,1,100000.0,99999.893565,-0.106435
8,AKAM,ok,130,0,100000.0,100000.000000,0.000000
9,ALGM,ok,130,0,100000.0,100000.000000,0.000000


Total trades: 260


,symbol,trades,wins,total_pnl,avg_pnl,avg_return_pct,win_rate
0,MSCI,2,1,6.060270,3.030135,NaN,50.0
1,ANAB,1,1,5.243335,5.243335,NaN,100.0
2,AAOI,2,1,2.910280,1.455140,NaN,50.0
3,MOH,1,1,2.507865,2.507865,NaN,100.0
4,ZEPP,1,1,2.413555,2.413555,NaN,100.0
5,ITW,2,1,2.330550,1.165275,NaN,50.0
6,DVA,1,1,2.210360,2.210360,NaN,100.0
7,TECX,2,1,2.202270,1.101135,NaN,50.0
8,ARW,1,1,1.914185,1.914185,NaN,100.0
9,SAP,2,1,1.861625,0.930812,NaN,50.0


,symbol,trade_day,entry_time,exit_time,entry_price,exit_price,size,pnl,return_pct,exit_reason,daily_rvol,daily_atrs_traded,stop_low_of_day
0,ROP,2026-01-27,2026-01-27 10:51:00,2026-01-27 11:00:00,None,None,None,-5.067205,NaN,ema5_cross_below_ema9,4.134557,5.277048,345.9300
1,ATEX,2026-01-27,2026-01-27 11:09:00,2026-01-27 12:03:00,None,None,None,-0.226720,NaN,ema5_cross_below_ema9,4.379272,3.753230,23.0300
2,SUPX,2026-01-27,2026-01-27 11:33:00,2026-01-27 11:48:00,None,None,None,-0.284050,NaN,ema5_cross_below_ema9,3.969422,2.450210,14.0000
3,GM,2026-01-27,2026-01-27 11:51:00,2026-01-27 11:57:00,None,None,None,-0.293340,NaN,ema5_cross_below_ema9,2.715012,3.323832,82.4100
4,GLW,2026-01-27,2026-01-27 11:54:00,2026-01-27 13:00:00,None,None,None,0.743895,NaN,ema5_cross_below_ema9,4.226986,4.502339,100.9700
5,APH,2026-01-27,2026-01-27 12:03:00,2026-01-27 12:06:00,None,None,None,-0.385040,NaN,ema5_cross_below_ema9,2.123926,2.048138,157.4500
6,GGB,2026-01-27,2026-01-27 12:09:00,2026-01-27 12:12:00,None,None,None,-0.014155,NaN,ema5_cross_below_ema9,2.025783,2.145043,4.4250
7,GGB,2026-01-27,2026-01-27 12:15:00,2026-01-27 12:24:00,None,None,None,-0.019140,NaN,ema5_cross_below_ema9,2.025783,2.145043,4.4250
8,SATL,2026-01-27,2026-01-27 13:24:00,2026-01-27 15:18:00,None,None,None,0.269030,NaN,ema5_cross_below_ema9,2.158189,2.292891,4.4800
9,UPS,2026-01-27,2026-01-27 13:30:00,2026-01-27 13:42:00,None,None,None,-0.470570,NaN,ema5_cross_below_ema9,2.666798,3.252639,104.7500
